# Super modèle — GPT taille GPT-2 sur TPU, session 1/2

In [ ]:
import subprocess, glob, shutil


def run(cmd: str) -> None:
    """Lance une commande en affichant sa sortie ; une erreur arrête le notebook (et la session)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'échec ({p.returncode}) : {cmd}')


for name in ('train_tpu.py', 'prepare_data.py'):
    shutil.copy(glob.glob(f'/kaggle/input/**/{name}', recursive=True)[0], f'/kaggle/working/{name}')

In [ ]:
run("pip install -q torch==2.9.0 --index-url https://download.pytorch.org/whl/cpu && pip install -q torch_xla==2.9.0 libtpu==0.0.21.1")
run("python -c \"import torch, torch_xla, torch_xla.runtime as xr; print(torch.__version__, torch_xla.__version__, xr.device_type(), xr.global_runtime_device_count()); assert xr.device_type() == 'TPU'\"")

In [ ]:
# Vitesse de chaque taille sur un cœur (≈ 10 min), puis le choix : GPT-2 medium si les 8 cœurs dépassent
# ~110 000 tokens/s (assez pour ~3 G tokens en 7 h), sinon GPT-2 small.
import json, os
run('cd /kaggle/working && python train_tpu.py bench')
bench = json.load(open('/kaggle/working/bench.json'))['bench']
print(bench)
ok = [b for b in bench if b.get('ok')]
medium = max((b for b in ok if b['model'] == 'medium'), key=lambda b: b['tokens_per_s_per_core'], default=None)
if medium and medium['tokens_per_s_per_core'] * 8 >= 110_000:
    MODEL, MICRO = 'medium', medium['micro_batch']
else:
    MODEL, MICRO = 'small', 16
print('choix :', MODEL, MICRO)

In [ ]:
run(f'cd /kaggle/working && SESSION=1 MODEL={MODEL} MICRO_BATCH={MICRO} HOURS_PRETRAIN=7.2 '
    'TOTAL_PRETRAIN_HOURS=7.2 HOURS_FINETUNE=0.4 python train_tpu.py train')

In [ ]:
run('rm -f /kaggle/working/train_tpu.py /kaggle/working/prepare_data.py; ls -la /kaggle/working /kaggle/working/checkpoints || true')